In [1]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

In [3]:
pdf_path = r"C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF.pdf"
pdf_path = r"TEXT_PDF.pdf"

In [ ]:
# ==============================
# 1. EXTRACT TEXT
# ==============================
helper = Helper()
text_data = helper.get_pdf_text(pdf_path)
print("TEXT SAMPLE:", text_data[0][:10])


In [ ]:
# ==============================
# 2. GET ALL BLOCKS + IMAGES
# ==============================
helper = Helper()
all_data = helper.get_all_pdf_data(pdf_path)
pprint.pprint(all_data[0])
# print(all_data[0])

In [ ]:
# ==============================
# 3. CLIPPED DATA (EDIT BBOX)
# ==============================
helper = Helper()
bboxes = [
    (0, 0, 300, 400),
    (100, 200, 400, 600)
]

try:
    clipped = helper.get_clipped_data(pdf_path, bboxes)
    print("CLIPPED SAMPLE:", clipped[0])
except Exception as e:
    print("Clipping skipped:", e)


In [ ]:
# ==============================
# 4. DRAW LINES + RECTS
# ==============================
lines = [
    ((50, 50), (300, 50)),
    ((100, 100), (400, 100))
]
rects = [(50, 50, 200, 300)]
pages = [1]

pdf_path = r"STR.pdf"
output_draw = pdf_path.replace(".pdf", "_drawn.pdf")
helper.draw_lines_on_pdf( pdf_path,lines,rects,pages,output_draw)



Modified PDF saved to: C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF_drawn.pdf


In [5]:
# ==============================
# 5. LINE BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"STR_bbox_mask.pdf"
output_path =helper.draw_boundaries_on_lines(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['STR_bbox_mask_line_hltd.pdf']>

In [5]:
# ==============================
# 6. BLOCK BOUNDARIES
# ==============================
helper = Helper()
output_path = helper.draw_boundaries_on_pdf(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['SAMPLE_block_highlighted.pdf']>

In [3]:
# ==============================
# 7. SPAN BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"KOT_bbox_mask.pdf"
output_path = helper.draw_span_boundaries(pdf_path)
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['KOT_bbox_mask_span_hltd.pdf']>

In [4]:
# ==============================
# 8. BBOX ONLY TEXT
# ==============================
helper = Helper()
pdf_path = r"STR.pdf"
output_path = helper.mask_outside_bboxes(pdf_path,[(4.1, 116.3, 268.64, 724.84)])
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['STR_bbox_mask.pdf']>

In [1]:
from app.utils import Helper

helper = Helper()
pdf_path = r"MAST.pdf"

pairs = [
    (
        (37.29, 100.68, 768.77, 472.33), 
        [382.83, 459.27, 538.82, 614.64, 689.84]
    )
]

helper.extract_sections_from_pairs(pdf_path,pairs)


['output\\p0_b0_c0.png',
 'output\\p0_b0_c1.png',
 'output\\p0_b0_c2.png',
 'output\\p0_b0_c3.png',
 'output\\p0_b0_c4.png',
 'output\\p0_b0_c5.png']

In [4]:
import fitz
import easyocr
import cv2
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor

# ---------------- CONFIG ----------------
pdf_path = "output.pdf"

pairs = [
    (
        (21.0, 2.0, 934.0, 469.0), 
        [23.0, 447.0, 570.0, 679.0, 769.0, 851.0, 927.0]
    )
]

scale = 2
out_dir = "sections"
os.makedirs(out_dir, exist_ok=True)

reader = easyocr.Reader(['en'])


# ---------------- STEP 1: EXTRACT SECTIONS ----------------
def extract_sections_from_pairs(pdf_path, bbox_line_pairs, scale):
    doc = fitz.open(pdf_path)

    section_meta = []   # (image_path, x_offset, y_offset)

    for page_idx, page in enumerate(doc):

        for b_idx, (bbox, v_lines) in enumerate(bbox_line_pairs):
            x0, y0, x1, y1 = bbox

            xs = [x0] + sorted(v_lines) + [x1]

            for i in range(len(xs) - 1):
                sx0 = xs[i]
                sx1 = xs[i + 1]

                rect = fitz.Rect(sx0, y0, sx1, y1)

                pix = page.get_pixmap(
                    matrix=fitz.Matrix(scale, scale),
                    clip=rect
                )

                path = os.path.join(out_dir, f"p{page_idx}_b{b_idx}_c{i}.png")
                pix.save(path)

                section_meta.append({
                    "path": path,
                    "x_offset": sx0,   # VERY IMPORTANT
                    "y_offset": y0
                })

    doc.close()
    return section_meta


# ---------------- STEP 2: OCR WORKER ----------------
def ocr_section(meta):
    img = cv2.imread(meta["path"])

    result = reader.readtext(img)

    data = []
    for bbox, text, prob in result:
        pts = np.array(bbox)

        # ✅ convert to global coordinates
        pts = pts / scale
        pts[:, 0] += meta["x_offset"]
        pts[:, 1] += meta["y_offset"]

        data.append((pts.tolist(), text, prob))

    return data


# ---------------- STEP 3: RUN OCR ----------------
section_meta = extract_sections_from_pairs(pdf_path, pairs, scale)

all_results = []

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(ocr_section, section_meta)

for res in results:
    all_results.extend(res)


# ---------------- STEP 4: SORT RESULTS ----------------
def get_top_left(item):
    bbox = item[0]
    x = min(p[0] for p in bbox)
    y = min(p[1] for p in bbox)
    return (y, x)

all_results = sorted(all_results, key=get_top_left)


# ---------------- STEP 5: BUILD SEARCHABLE PDF ----------------
def build_searchable_pdf(pdf_path, all_results, scale):
    doc = fitz.open(pdf_path)
    out_doc = fitz.open()

    for page in doc:
        rect = page.rect

        new_page = out_doc.new_page(width=rect.width, height=rect.height)

        # ✅ insert original image
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
        new_page.insert_image(new_page.rect, pixmap=pix)

        # ✅ overlay text layer
        for bbox, text, prob in all_results:
            if prob < 0.3:
                continue

            pts = np.array(bbox)

            x = int(pts[:, 0].min())
            y = int(pts[:, 1].min())

            y_pdf = rect.height - y

            # estimate font size
            h_text = pts[:, 1].max() - pts[:, 1].min()
            fontsize = max(6, int(h_text * 0.6))

            new_page.insert_text(
                (x, y_pdf),
                text,
                fontsize=fontsize,
                render_mode=3   # ✅ invisible text
            )

    out_doc.save("searchable_output.pdf")
    print("✅ OCR searchable PDF created")


# ---------------- STEP 6: EXECUTE ----------------
build_searchable_pdf(pdf_path, all_results, scale)


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


✅ OCR searchable PDF created


In [ ]:
# ==============================
# 9. SINGLE BBOX DRAW
# ==============================
bbox = (100, 100, 400, 400)
helper = Helper()
helper.draw_bboxes_on_pdf(pdf_path, bbox)



In [3]:
# lines = [
#     ((110, 0), (110, 812)),# Vertical line
#     ((0, 350), (812, 350)),
#     ((570, 0), (570, 812))
# ]
from pathlib import Path
from app.utils import Helper

lines  = line_content
pages = [2]
bboxes = []
sample_path = r"C:\Users\kaustubh.keny\Downloads\apollo_table 2.pdf"
file_name = Path(sample_path).name


Helper.draw_lines_on_pdf(sample_path, lines, bboxes, pages, file_name.replace(".pdf","_line.pdf"))

Modified PDF saved to: apollo_table 2_line.pdf


In [ ]:
import fitz
import pytesseract
from PIL import Image
import io, re

def get_proper_fund_names(path: str):
    title = {}
    pattern ="((?:LI?i?C|BSE|BANK|SMALL|HEALTH|MNEY|[aA]n\\s*open).*?(?:FUND|Path|ETF|FTF|EOF|FOF|PLAN|SAVER|tax saving scheme|small cap stocks)\\s*(?:FUND\\s*OF\\s*FUND)?)"
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            clip = fitz.Rect(300, 0, 595, 80)
            pix = page.get_pixmap(clip=clip, dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            cleaned = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            if matches := re.findall(pattern, cleaned, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _])
                print(f"{pgn}:matched {matches[0]}")
            # print(f"[OCR] Page {pgn}: {cleaned}")
            # if cleaned:
            #     title[pgn] = cleaned
    return title
path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Documents\MUTUAL FUND FACTSHEET FY19-25\2021_changed\LIC Mutual Fund\25_31-Dec-21_FS.pdf"
title = get_proper_fund_names(path)

In [7]:
import re, fitz

def get_proper_fund_names(path: str, pattern:str,clip:tuple):
    title = {} 
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            text = " ".join(page.get_text("text", clip = clip).split("\n"))
            text = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            # print(f"{pgn}:-{text}")
            if matches := re.findall(pattern, text, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _ ])
                print(pgn,matches[0])
    return title
path = r"35_30-Jun-26_FS.pdf"
pattern = "((?:SBI).+?(?:FUND|F[Oo]F)(?:.+?PLAN)?)"
clip = (100, 0, 500, 55)
title = get_proper_fund_names(path,pattern,clip)

12 SBI LARGE CAP FUND
13 SBI LARGE & MIDCAP FUND
14 SBI QUANT FUND
15 SBI DIVIDEND YIELD FUND
16 SBI ELSS TAX SAVER FUND
17 SBI FLEXICAP FUND
18 SBI FOCUSED FUND
19 SBI MULTICAP FUND
20 SBI CONTRA FUND
21 SBI MIDCAP FUND
22 SBI SMALL CAP FUND
23 SBI HEALTHCARE OPPORTUNITIES FUND
24 SBI BANKING & FINANCIAL SERVICES FUND
25 SBI TECHNOLOGY OPPORTUNITIES FUND
26 SBI INFRASTRUCTURE FUND
27 SBI PSU FUND
28 SBI COMMA FUND
29 SBI EQUITY MINIMUM VARIANCE FUND
30 SBI ESG EXCLUSIONARY STRATEGY FUND
31 SBI MNC FUND
32 SBI CONSUMPTION OPPORTUNITIES FUND
33 SBI QUALITY FUND
34 SBI AUTOMOTIVE OPPORTUNITIES FUND
35 SBI ENERGY OPPORTUNITIES FUND
36 SBI INNOVATIVE OPPORTUNITIES FUND
37 SBI US SPECIFIC EQUITY ACTIVE FOF
40 SBI ARBITRAGE OPPORTUNITIES FUND
41 SBI CONSERVATIVE HYBRID FUND
42 SBI EQUITY SAVINGS FUND
43 SBI BALANCED ADVANTAGE FUND
44 SBI MULTI ASSET ALLOCATION FUND
45 SBI EQUITY HYBRID FUND
46 SBI CHILDRENS FUND-SAVINGS PLAN
47 SBI CHILDRENS FUND-INVESTMENT PLAN
48 SBI RETIREMENT BENEFIT FUN

In [ ]:
import pandas as pd
import os

from app.parse_table import TableParser


parser = TableParser()

path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Aditya Birla Sun Life Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Bajaj Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\HSBC Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Navi MutuaL Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Shriram Mutual fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Bandhan Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Bank of India Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Baroda BNP Paribas Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\DSP Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Edelweiss Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\HDFC Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Helios Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Invesco Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\ITI Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Kotak Mutal Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\LIC Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Mahindra Manulife Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Mirae Asset Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Motilal Oswal Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Nippon India Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\NJ Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Oldbridge Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\PGIM India Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\SAMCO Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\SBI Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Sundaram Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Franklin Templeton Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Taurus Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Whiteoak capital Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\360 ONE Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\ICICI Mutual Fund"
path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Union Mutual Fund"
all_data = []

for file in os.listdir(path):
    if not file.endswith(".pdf"):
        continue
    
    pdf_path = os.path.join(path,file)
    
    print(file)
    df = parser.extract_tables_from_pdf(path=pdf_path,pages=None)
    # print(df.shape)
    
    old_cols = list(df.columns)
    
    df["filename"] = file
    
    new_cols = ["filename"] + old_cols
    df = df.reindex(columns = new_cols)
    all_data.append(df)
    
d1 = pd.concat(all_data, axis=0, ignore_index=True)
d1.to_excel("UNION_FINAL.xlsx")

0